In [12]:
import numpy as np
import pickle
import casadi as ca
import time

num_var = 100
num_ineq = 50
num_eq = 50
num_examples = 2
seed = 2025

print("Nonsmooth nonconvex SOCP problem with {} variables, {} inequalities, {} equalities and {} examples".format(num_var, num_ineq, num_eq, num_examples))
np.random.seed(seed)
Q = np.diag(np.random.rand(num_var)*0.5)
p = np.random.uniform(-1, 1, num_var)
A = np.random.uniform(-1, 1, size=(num_eq, num_var))
X = np.random.uniform(-1, 1, size=(num_examples, num_eq))
XL = X.min(axis=0)
XU = X.max(axis=0)

L = np.ones((num_var))*-5
U = np.ones((num_var))*5
x0 = np.random.uniform(-1, 1, size=(num_var))
G = []
h = []
C = []
d = []
for i in range(num_ineq):
    G.append(np.random.uniform(-1, 1, size=(num_ineq, num_var)))
    h.append(np.random.uniform(-1, 1, size=(num_ineq)))
    C.append(np.random.uniform(-1, 1, size=(num_var)))
    d.append(np.linalg.norm(G[i] @ x0 + h[i], 2) - C[i].T @ x0)
data = {'Q':Q,
        'p':p,
        'A':A,
        'X':X,
        'G':np.array(G),
        'h':np.array(h),
        'C':np.array(C),
        'd':np.array(d),
        'YL':L,
        'YU':U,
        'XL':XL,
        'XU':XU,
        'Y':[]}

Nonsmooth nonconvex SOCP problem with 100 variables, 50 inequalities, 50 equalities and 2 examples


In [24]:
Y = []
for n in range(num_examples):
    Xi = X[n]
    y = ca.MX.sym('y_var', num_var)
    t = ca.MX.sym('t_var')

    obj_func = 0.5 * ca.mtimes(y.T, ca.mtimes(Q, y)) + ca.dot(p, ca.sin(y)) + 0.1*t

    eq_constraints = A @ y - Xi
    soc = ca.dot(y, y) - t**2
    ineq_constraints = []
    for i in range(num_ineq):
        ineq_constraints.append(ca.norm_2(G[i] @ ca.cos(y) + h[i]) - (ca.dot(C[i], y) + d[i]))
    ineq_constraints.append(soc)
    ineq_constraints = ca.vertcat(*ineq_constraints)
    
    nlp = {'x': ca.vertcat(y, t), 'f': obj_func, 'g': ca.vertcat(eq_constraints, ineq_constraints)}
    # opts = {'ipopt.print_level': 0, 'print_time': 0, }
    opts = {
        "ipopt.print_level": 0,
        "print_time": 0,

        # ---- VERY LOOSE TOLERANCES ----
        "ipopt.tol": 1e-1,
        "ipopt.constr_viol_tol": 1e-1,
        "ipopt.dual_inf_tol": 1e-1,
        "ipopt.compl_inf_tol": 1e-1,

        # Accept early termination
        "ipopt.acceptable_tol": 1e0,
        "ipopt.acceptable_constr_viol_tol": 1e0,
        "ipopt.acceptable_dual_inf_tol": 1e0,
        "ipopt.acceptable_compl_inf_tol": 1e0,

        # Stop as soon as acceptable is reached
        "ipopt.acceptable_iter": 1,
    }
    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)
    # Define bounds for variables and constraints
    lbg = np.concatenate([np.zeros(num_eq), -np.inf * np.ones(num_ineq+1)])
    ubg = np.concatenate([np.zeros(num_eq), np.zeros(num_ineq+1)])
    lbx = np.concatenate([L, [0]])
    ubx = np.concatenate([U, [np.inf]])

    start_time = time.time()
    res = solver(lbg=lbg, ubg=ubg, lbx=lbx, ubx=ubx)
    python_wall = time.time() - start_time

    stats = solver.stats()

    solver_wall = sum(
        stats[k] for k in stats if k.startswith("t_wall_")
    )

    print(f"Python wall time : {python_wall:.4f} s")
    print(f"IPOPT wall time  : {solver_wall:.4f} s")

    # check if the solver converged
    if solver.stats()['success']:
        sol_x = res['x'].full().flatten()
        Y.append(sol_x[:-1])
    else:
        print("Solver failed to converge")
        break

    print("Example {}: Objective value: {}".format(n, res['f'].full().flatten()[0]))

data['Y'] = np.array(Y)


i = 0
det_min = 0
best_partial = 0
while i < 1000:
    np.random.seed(i)
    partial_vars = np.random.choice(num_var, num_var - num_eq, replace=False)
    other_vars = np.setdiff1d(np.arange(num_var), partial_vars)
    _, det = np.linalg.slogdet(A[:, other_vars])
    if det>det_min:
        det_min = det
        best_partial = partial_vars
    i += 1
print('best_det', det_min)
data['best_partial'] = best_partial


# with open("datasets/nonsmooth_nonconvex/socp/random{}_socp_dataset_var{}_ineq{}_eq{}_ex{}".format(seed, num_var, num_ineq, num_eq, num_examples), 'wb') as f:
#     pickle.dump(data, f)

Python wall time : 3.3126 s
IPOPT wall time  : 3.0719 s
Example 0: Objective value: 21.73689217966514
Python wall time : 3.7316 s
IPOPT wall time  : 3.4634 s
Example 1: Objective value: 20.458163602003005
best_det 50.35299906971193


In [ ]:
Python wall time : 4.1379 s
IPOPT wall time  : 3.8211 s
Example 0: Objective value: 21.830696173485194
Python wall time : 5.1073 s
IPOPT wall time  : 4.7339 s
Example 1: Objective value: 20.49899542514615
best_det 50.35299906971193

In [ ]:
Python wall time : 5.9450 s
IPOPT wall time  : 5.5313 s
Example 0: Objective value: 9.150703123482039
Python wall time : 7.8945 s
IPOPT wall time  : 7.3162 s
Example 1: Objective value: -0.2152281530008182
best_det 50.35299906971193

SyntaxError: invalid syntax (4173506394.py, line 1)

In [ ]:
# print all input and output data
for key, value in data.items():
    print(key, value)

Q [[0.06774408 0.         0.         ... 0.         0.         0.        ]
 [0.         0.44392585 0.         ... 0.         0.         0.        ]
 [0.         0.         0.46630282 ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.30484921 0.         0.        ]
 [0.         0.         0.         ... 0.         0.13594885 0.        ]
 [0.         0.         0.         ... 0.         0.         0.356382  ]]
p [-0.26797082 -0.44038608  0.76300533 -0.66656702 -0.79830711  0.94824372
  0.91290407 -0.74939026 -0.73698738 -0.31904344  0.0010889   0.69516202
  0.13636374 -0.56713024  0.24288352 -0.22757172 -0.23043851 -0.62175557
  0.91478575 -0.75375373 -0.82693273  0.20347321  0.19183125 -0.74784845
  0.2300497  -0.34523786 -0.30941384 -0.31858224 -0.75522649  0.89848064
 -0.29221899 -0.01913345 -0.87505192 -0.02145402 -0.30016887  0.92214443
 -0.47202405  0.47158661 -0.00250927  0.85352621  0.3526678  -0.85453358
 -0.47796119 -0.62643053  0.28876086 -0.8

In [17]:
import numpy as np
from scipy.optimize import minimize, NonlinearConstraint, Bounds


# -----------------------------
# Objective
# -----------------------------
def objective(z):
    y = z[:-1]
    t = z[-1]
    return 0.5 * y @ Q @ y + p @ np.sin(y) + 0.1 * t

# -----------------------------
# Constraints
# -----------------------------
def equality_constraint(z, x):
    y = z[:-1]
    return A @ y - x

def soc_constraint(z):
    y = z[:-1]
    t = z[-1]
    return np.dot(y, y) - t**2

def nonsmooth_constraints(z):
    y = z[:-1]
    return np.array([
        np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        for i in range(num_ineq)
    ])


# -----------------------------
# Solve for each X
# -----------------------------
Y2 = []

for n in range(num_examples):
    x = X[n]

    eq_con = NonlinearConstraint(
        lambda z, x=x: equality_constraint(z, x),
        lb=np.zeros(num_eq),
        ub=np.zeros(num_eq)
    )

    soc_con = NonlinearConstraint(
        soc_constraint,
        lb=-np.inf,
        ub=0.0
    )

    nonsmooth_con = NonlinearConstraint(
        nonsmooth_constraints,
        lb=-np.inf * np.ones(num_ineq),
        ub=np.zeros(num_ineq)
    )

    bounds = Bounds(
        np.concatenate([L, [0.0]]),
        np.concatenate([U, [np.inf]])
    )

    z0 = np.concatenate([np.random.uniform(L, U), [1.0]])

    res = minimize(
        objective,
        z0,
        method="SLSQP",
        bounds=bounds,
        constraints=[eq_con, soc_con, nonsmooth_con],
        options={"ftol": 1e-6}
    )

    if not res.success:
        print(f"Failed at sample {n}: {res.message}")
        break

    Y2.append(res.x[:-1])

    print(f"Sample {n}, obj = {res.fun:.6f}")

Y2 = np.array(Y2)
print("Finished SciPy optimization")


Sample 0, obj = 2.083667
Sample 1, obj = 1.887357
Finished SciPy optimization


In [ ]:
# print solutions
print("Y:", Y)
print("Y2:", Y2)
print("Difference between Y and Y2:", np.linalg.norm(Y - Y2))

Y: [array([ 1.07294578,  0.63436048, -1.14111867,  1.63723902,  1.60368882,
       -1.99441533, -1.27029699,  1.02628696,  0.39818766,  0.93896749,
        1.07260082, -1.24004727,  1.04668948,  1.57027249, -1.78613409,
        0.3708503 ,  1.47560166, -1.0428162 ,  1.28505918,  1.31524225,
       -1.09439858, -1.45701147,  0.67717941,  1.37438883, -0.95698122,
        1.20670668, -0.73582651,  0.94622208,  1.18604969,  2.08020434,
        1.27058969, -1.44594744,  1.29667862,  1.84331384,  1.86530976,
        1.04800623,  1.76295487, -1.59443681, -1.68917414, -1.34719397,
       -0.99853892, -0.47675053,  0.24318357,  0.81225489,  0.80131874,
        0.78270118,  1.72683141, -1.14284029, -1.2889654 ,  1.03174186,
        1.26374   ,  1.65312599,  1.07012834,  0.79485612, -1.48730222,
       -0.06369301, -0.60909088, -1.38156478, -0.91951777, -0.75437596,
       -1.04818946, -1.2675111 , -1.29249588,  1.35526805, -1.3759763 ,
       -0.92253949,  1.2139995 , -1.71479315,  1.35383878, -

ValueError: operands could not be broadcast together with shapes (10,100) (3,100) 

In [18]:
def constraint_violation(y, t, x):
    # Equality violation
    eq_violation = np.linalg.norm(A @ y - x, ord=2)

    # Inequality violations
    ineq_vals = []
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        ineq_vals.append(max(0.0, gi))

    soc = np.dot(y, y) - t**2
    ineq_vals.append(max(0.0, soc))

    ineq_vals = np.array(ineq_vals)

    # Bound violations
    lb_violation = np.maximum(0.0, L - y)
    ub_violation = np.maximum(0.0, y - U)
    t_violation  = max(0.0, -t)

    bound_violation = np.linalg.norm(
        np.concatenate([lb_violation, ub_violation, [t_violation]]),
        ord=2
    )

    return {
        "eq_l2": eq_violation,
        "ineq_max": ineq_vals.max(),
        "ineq_l2": np.linalg.norm(ineq_vals, ord=2),
        "bound_l2": bound_violation
    }
def grad_objective(y, t):
    grad_y = Q @ y + p * np.cos(y)
    grad_t = np.array([0.1])
    return np.concatenate([grad_y, grad_t])
def jacobian_eq():
    J = np.zeros((num_eq, num_var + 1))
    J[:, :num_var] = A
    return J

def jacobian_soc(y, t):
    J = np.zeros(num_var + 1)
    J[:num_var] = 2 * y
    J[-1] = -2 * t
    return J

def jacobian_nonsmooth(y, i):
    v = G[i] @ np.cos(y) + h[i]
    norm_v = np.linalg.norm(v)

    if norm_v < 1e-8:
        return np.zeros(num_var + 1)

    J = np.zeros(num_var + 1)
    J[:num_var] = (
        -(G[i].T @ (v / norm_v)) * np.sin(y) - C[i]
    )
    return J
def kkt_residual(y, t, x):
    grad_f = grad_objective(y, t)

    # Active constraints
    J = []
    rhs = -grad_f

    # Equality constraints
    J.append(jacobian_eq())

    # Active inequalities
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        if gi > -1e-6:
            J.append(jacobian_nonsmooth(y, i)[None, :])

    soc = np.dot(y, y) - t**2
    if soc > -1e-6:
        J.append(jacobian_soc(y, t)[None, :])

    if not J:
        return np.linalg.norm(grad_f)

    J = np.vstack(J)

    # Least-squares multipliers
    try:
        lam, *_ = np.linalg.lstsq(J.T, rhs, rcond=None)
        res = grad_f + J.T @ lam
        return np.linalg.norm(res)
    except np.linalg.LinAlgError:
        return np.inf

In [21]:
# y = Y2[-1]      # Scipy solution
# t = res.x[-1]
# x = X[-1]

y = Y[-1]      # IPOPT solution
t = sol_x[-1]
x = X[-1]



viol = constraint_violation(y, t, x)
kkt  = kkt_residual(y, t, x)

print("Constraint violation:", viol)
print("KKT residual:", kkt)

Constraint violation: {'eq_l2': np.float64(1.5878130082852256e-14), 'ineq_max': np.float64(0.0), 'ineq_l2': np.float64(0.0), 'bound_l2': np.float64(0.0)}
KKT residual: 3.596340847333199
